<a href="https://colab.research.google.com/github/Shruthisajeevan/projectdemo/blob/master/1text_sumirization_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y transformers
!pip install transformers==4.41.2 sentencepiece accelerate

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)


In [ ]:
!pip install nltk transformers torch sentencepiece

1

In [1]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: Tesla T4


In [3]:
!pip uninstall sentence-transformers -y

Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3


In [2]:
import re
import heapq
from collections import defaultdict
import nltk
import torch
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from transformers import pipeline

# Ensure the transformers library is up-to-date and correctly installed.
nltk.download('punkt')
nltk.download('stopwords')
# --------------------------------------------------
# DEVICE SETUP (Auto GPU if available)
# --------------------------------------------------

device = 0 if torch.cuda.is_available() else -1

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# LOAD TRANSFORMER MODEL (BART)
# --------------------------------------------------

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device
)

# --------------------------------------------------
# EXTRACTIVE SUMMARIZATION
# --------------------------------------------------

def extractive_summarize(text, num_sentences=4):
    if not text.strip():
        return ""

    sentences = sent_tokenize(text)
    stop_words = set(stopwords.words("english"))
    freq = defaultdict(int)

    # Word frequency
    for word in word_tokenize(text.lower()):
        if word.isalnum() and word not in stop_words:
            freq[word] += 1

    if not freq:
        return " ".join(sentences[:num_sentences])

    max_freq = max(freq.values())

    # Score sentences
    scores = {}
    for i, sent in enumerate(sentences):
        words = word_tokenize(sent.lower())
        score = sum(freq[w] / max_freq for w in words if w in freq)

        # Position bonus
        position_bonus = max(0, 1.5 - i * 0.3)
        scores[sent] = score * (1 + position_bonus)

    # Select top sentences
    top_sentences = heapq.nlargest(num_sentences, scores, key=scores.get)
    top_sentences.sort(key=lambda s: sentences.index(s))

    return " ".join(top_sentences)

# --------------------------------------------------
# ABSTRACTIVE SUMMARIZATION
# --------------------------------------------------

def abstractive_summarize(text):
    summary = summarizer(
        text,
        max_length=130,
        min_length=40,
        do_sample=False
    )[0]["summary_text"]

    return summary.strip()

# --------------------------------------------------
# MAIN SUMMARIZATION FUNCTION
# --------------------------------------------------

def summarize_article(article):
    result = {}

    result["original_words"] = len(article.split())

    extractive = extractive_summarize(article)
    result["extractive"] = extractive

    abstractive = abstractive_summarize(article)
    result["abstractive"] = abstractive

    return result

# --------------------------------------------------
# DEMO ARTICLE (You Can Replace This)
# --------------------------------------------------

article = """
The Indian Premier League (IPL) 2026 season has begun with explosive performances across multiple venues.
Royal Challengers Bengaluru started their campaign with a thrilling victory over Chennai Super Kings.
Virat Kohli played a match-winning knock while bowlers dominated the death overs.
Several records have already been broken including the highest team total and fastest fifty.
Experts predict another high-scoring season due to batting-friendly pitches.
Franchises are focusing on aggressive middle-order strategies and improved death bowling.
Viewership is expected to cross 550 million globally.
"""

print("\n" + "="*80)
print("ORIGINAL ARTICLE:\n")
print(article)
print("\nWord Count:", len(article.split()))

print("\n" + "="*80)

summary = summarize_article(article)

print("\nEXTRACTIVE SUMMARY:\n")
print(summary["extractive"])

print("\n" + "="*80)

print("\nABSTRACTIVE SUMMARY:\n")
print(summary["abstractive"])

print("\n" + "="*80)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


GPU Available: True
GPU Name: Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Your max_length is set to 130, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)



ORIGINAL ARTICLE:


The Indian Premier League (IPL) 2026 season has begun with explosive performances across multiple venues.
Royal Challengers Bengaluru started their campaign with a thrilling victory over Chennai Super Kings.
Virat Kohli played a match-winning knock while bowlers dominated the death overs.
Several records have already been broken including the highest team total and fastest fifty.
Experts predict another high-scoring season due to batting-friendly pitches.
Franchises are focusing on aggressive middle-order strategies and improved death bowling.
Viewership is expected to cross 550 million globally.


Word Count: 83


EXTRACTIVE SUMMARY:


The Indian Premier League (IPL) 2026 season has begun with explosive performances across multiple venues. Royal Challengers Bengaluru started their campaign with a thrilling victory over Chennai Super Kings. Virat Kohli played a match-winning knock while bowlers dominated the death overs. Several records have already been broken inc